# How to: first steps in FELiCS 

[Link to installation guide]
How to import FELiCS...

1. Load a mesh file from gmsh, read h5 files, add two fields together and export them.

TODOS:
- debug import export

(- write static method in new FEMSpaces class (new name?))


SCRIPT:
Create felics mesh from file        check
Create P2 space on mesh             check
Import h5 files with function values using field
Add fields together
Export


In [6]:
from FELiCS.SpaceDisc.FELiCSMesh import FELiCSMesh
from FELiCS.SpaceDisc.DefineFEMSpaces import FEMSpacesClass
from FELiCS.Fields.Field import Field


meshFileName = "./square_mesh.msh"
coordinateSystem = "Cartesian"

# Create a FELiCS mesh from the gmsh file
mesh = FELiCSMesh(coordinateSystem=coordinateSystem, meshFileName=meshFileName)

# Will be replaced later
from dolfinx.fem import functionspace
P2Space = functionspace(mesh.dolfinxMesh, ("Lagrange", 2))

Field1 = Field(P2Space, mesh=mesh)
Field1.importH5("function_values_quadratic")
Field2 = Field(P2Space, mesh=mesh)
Field2.importH5("function_values_sine")

Field3 = Field1 + Field2

Field3.exportH5("field_sum")







Info     | FELiCSMesh.py          | __init__                   (line 28  ) : Opening mesh file: ./square_mesh.msh
Info     | FELiCSMesh.py          | __init__                   (line 36  ) : Mesh contains 513 nodes and 1024 elements


In [1]:
import numpy as np

# Define some test functions to evaluate on the mesh
def test_function_1(x):
    """Simple quadratic function: f(x,y) = x^2 + y^2"""
    return x[0]**2 + x[1]**2

def test_function_2(x):
    """Sine wave function: f(x,y) = sin(2*pi*x) * cos(2*pi*y)"""
    return np.sin(2 * np.pi * x[0]) * np.cos(2 * np.pi * x[1])

def test_function_3(x):
    """Gaussian function: f(x,y) = exp(-(x-0.5)^2 - (y-0.5)^2)"""
    return np.exp(-((x[0] - 0.5)**2 + (x[1] - 0.5)**2))

# Get coordinates of all DOFs in the P2 space
dof_coordinates = P2Space.tabulate_dof_coordinates()
print(f"Number of DOFs: {len(dof_coordinates)}")
print(f"DOF coordinates shape: {dof_coordinates.shape}")

# Calculate function values at DOF coordinates
function_values_1 = np.zeros(len(dof_coordinates))
function_values_2 = np.zeros(len(dof_coordinates))

for i, coord in enumerate(dof_coordinates):
    function_values_1[i] = test_function_1(coord)
    function_values_2[i] = test_function_2(coord)

# Save to numpy files
np.save('function_values_quadratic.npy', function_values_1)
np.save('function_values_sine.npy', function_values_2)
np.save('dof_coordinates.npy', dof_coordinates)

print("Function values saved to .npy files:")
print("- function_values_quadratic.npy")
print("- function_values_sine.npy") 
print("- function_values_gaussian.npy")
print("- dof_coordinates.npy")


NameError: name 'P2Space' is not defined